# Study Habits and Academic Performance

A reproducible analysis of a **synthetic** student dataset. The goal is to demonstrate a complete data-science workflow: data generation, exploratory analysis, linear regression, held-out evaluation, and communication through visualizations.

> This project uses no real student records. All observations are simulated.

## Research question

To what extent are weekly study hours, attendance, prior GPA, tutoring access, and first-generation status associated with exam performance?

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from generate_data import generate_study_data
from analyze import DATA_PATH, FEATURES, build_summary, create_figures, fit_and_evaluate

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    generate_study_data().to_csv(DATA_PATH, index=False)

data = pd.read_csv(DATA_PATH)
data.head()

## Data quality and descriptive statistics

In [ ]:
print(f'Rows: {len(data):,}')
print(f'Missing values: {int(data.isna().sum().sum())}')
data[FEATURES + ['exam_score']].describe().round(2)

## Model and held-out evaluation

A linear regression model is trained on 75% of the data and evaluated on the remaining 25%. The split is fixed with `random_state=42` for reproducibility.

In [ ]:
model, metrics, _ = fit_and_evaluate(data)
summary = build_summary(data, model, metrics)
pd.Series(summary).to_frame('value')

## Visualizations

The first figure summarizes the score distribution. The second examines the relationship between study time and exam score, with tutoring status shown by color.

In [ ]:
create_figures(data, model)

display(Image(filename=str(PROJECT_ROOT / 'visualizations' / 'score_distribution.png')))

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'visualizations' / 'study_hours_vs_score.png')))

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'visualizations' / 'regression_coefficients.png')))

## Interpretation

- Study time has a positive association with exam score: the estimated coefficient is about **1.4 points per additional weekly study hour**, holding the other variables constant.
- Students with tutoring access score about **4.5 points higher on average** than those without access.
- The model explains roughly half of the held-out variation in scores (`R² ≈ 0.53`), so important factors remain unmeasured.

**Limitation:** Because the dataset is synthetic and generated from a specified model, the results demonstrate analytical workflow and interpretation rather than evidence about real students. No causal claim should be drawn.

## Reproduce

From the repository root:

```bash
pip install -r requirements.txt
python src/generate_data.py
python src/analyze.py
jupyter nbconvert --to notebook --execute notebooks/study_performance_analysis.ipynb --output study_performance_analysis.ipynb
```